# this notebook is for attempt of optimization

In [1]:
import torch
import torch.optim as optim
from torch.export import Dim
from pathlib import Path
from pet_breed_classifier.data import get_dataloaders
from pet_breed_classifier.model import get_model, get_criterion

In [2]:
root = Path("../data").resolve()
ft_lr_3e_minus5 = Path("../parameters/model2/parameters2.pt").resolve()
train_loader, val_loader, test_loader, classes = get_dataloaders(root)

device = "cpu"

model = get_model(len(classes), device)
model.eval()

model.load_state_dict(torch.load(ft_lr_3e_minus5, map_location=device))

<All keys matched successfully>

In [3]:
dummy = torch.randn(2, 3, 224, 224)
torch.onnx.export(model, (dummy), "model.onnx",
                  input_names=["image"], output_names=["logits"],
                  dynamic_shapes = {"x": {0:Dim("batch")}})

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...


W1029 10:58:57.391000 25384 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.14.1+cu126',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"image"<FLOAT,[batch,3,224,224]>
            ),
            outputs=(
                %"logits"<FLOAT,[batch,37]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[64,3,7,7]>{Tensor(...)},
                %"layer1.0.conv1.weight"<FLOAT,[64,64,3,3]>{Tensor(...)},
                %"layer1.0.conv2.weight"<FLOAT,[64,64,3,3]>{Tensor(...)},
                %"layer1.1.conv1.weight"<FLOAT,[64,64,3,3]>{Tensor(...)},
                %"layer1.1.conv2.weight"<FLOAT,[64,64,3,3]>{Tensor(...)},
                %"layer2.0.conv1.weight"<FLOAT,[128,64,3,3]>{Tensor(...)},
                %"layer2.0.conv2.weight"<FLOAT,[128,128,3,3]>{Tensor(...)},
     

In [4]:
x = torch.randn(5, 3, 224, 224)

with torch.no_grad():
    torch_out = model(x).numpy()

print(torch_out.shape)

(5, 37)


In [5]:
import onnxruntime as ort

session = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
onnx_out = session.run(None, {input_name:x.numpy()})[0]

print(input_name, onnx_out.shape)

image (5, 37)


In [6]:
import numpy as np
print(np.abs(torch_out - onnx_out).max())
print(np.allclose(torch_out, onnx_out, atol=1e-4))
print((torch_out.argmax(1)==onnx_out.argmax(1)).all())

3.0994415e-06
True
True


In [7]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

m = onnx.load("model.onnx")
del m.graph.value_info[:]
onnx.save(m, "model_clean.onnx")

quantize_dynamic("model_clean.onnx", "model_int8.onnx", weight_type=QuantType.QInt8)

In [8]:
import os

path_pt = Path("../parameters/model2/parameters2.pt").resolve()
path_onnx = Path("model_clean.onnx").resolve()
path_int8onnx = Path("model_int8.onnx").resolve()

print(os.path.getsize(path_pt)/1e6)
print(os.path.getsize(path_onnx)/1e6)
print(os.path.getsize(path_int8onnx)/1e6)

44.861707
44.846085
11.275042


In [9]:
import torch
from pet_breed_classifier.benchmark import benchmark

x_torch = torch.randn(1, 3, 224, 224)
x_np = x_torch.numpy()

model_cpu = model.cpu().eval()
sess_fp32 = ort.InferenceSession(str(path_onnx), providers=["CPUExecutionProvider"])
sess_int8 = ort.InferenceSession(str(path_int8onnx), providers=["CPUExecutionProvider"])

with torch.no_grad():
    t_torch = benchmark(lambda x: model_cpu(x), x_torch)
t_fp32 = benchmark(lambda x: sess_fp32.run(None, {"image":x}), x_np)
t_int8 = benchmark(lambda x: sess_int8.run(None, {"image":x}), x_np)

print(f"PyTorch {t_torch:.1f} ms | ONNX FP32 {t_fp32:.1f} ms | ONNX INT8 {t_int8:1f} ms")

PyTorch 76.4 ms | ONNX FP32 60.0 ms | ONNX INT8 510.540348 ms


In [10]:
import importlib, pet_breed_classifier.benchmark as bm
importlib.reload(bm)
from pet_breed_classifier.benchmark import onnx_accuracy

acc_fp32 = onnx_accuracy(sess_fp32, test_loader)
acc_int8 = onnx_accuracy(sess_int8, test_loader)

print(f"ONNX FP32 {acc_fp32:.3f} | ONNX INT8 {acc_int8:.3f}")

ONNX FP32 0.898 | ONNX INT8 0.899


In [15]:
import pandas as pd
import os
results = pd.DataFrame({
    "Version" : ["PyTorch FP32", "ONNX FP32", "ONNX INT8"],
    "Size (MB)" : [os.path.getsize(p)/1e6 for p in (path_pt, path_onnx, path_int8onnx)],
    "CPU latency (ms/img)": [t_torch, t_fp32, t_int8],
    "Test Accuracy": [0.897, acc_fp32, acc_int8]
}).round(3)

print(results)

        Version  Size (MB)  CPU latency (ms/img)  Test Accuracy
0  PyTorch FP32     44.862                76.444          0.897
1     ONNX FP32     44.846                60.006          0.898
2     ONNX INT8     11.275               510.540          0.899


In [ ]:
import json
with open("../models/classes.json", "w") as f:
    json.dump(classes, f)

In [19]:
import json
with open("../reports/onnx_results.json", "w") as f:
    json.dump(results.to_dict(orient="records"), f)